[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/42_rloo_loss_solution.ipynb)

# 参考解答：留一法策略梯度损失（RLOO）

参考解答。

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✅ 参考解答

def rloo_loss(logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None) -> Tensor:
    """留一法策略梯度（RLOO）损失。

    logps: (B,) 序列级对数概率，或 (B, T) 词元级对数概率
    rewards: (B,) 每个采样回复的标量奖励
    group_ids: (B,) 整数组编号，相同编号表示同一个提示词／组
    mask: 可选，形状为 (B, T) 的词元掩码，1 表示计入，0 表示忽略
    返回：标量损失（Tensor）
    """
    if logps.dim() == 2:
        if mask is not None:
            mask_f = mask.to(dtype=logps.dtype, device=logps.device)
            seq_logps = (logps * mask_f).sum(dim=-1)
        else:
            seq_logps = logps.sum(dim=-1)
    elif logps.dim() == 1:
        seq_logps = logps
    else:
        raise ValueError('logps must have shape (B,) or (B, T)')

    rewards_detached = rewards.detach().to(dtype=seq_logps.dtype, device=seq_logps.device)
    advantages = torch.zeros_like(rewards_detached)

    for gid in group_ids.unique():
        group_mask = group_ids == gid
        r_g = rewards_detached[group_mask]
        k = r_g.numel()
        if k > 1:
            baseline = (r_g.sum() - r_g) / (k - 1)
            advantages[group_mask] = r_g - baseline

    return -(advantages.detach() * seq_logps).mean()

In [ ]:
# 示例
logps = torch.tensor([-0.5, -1.5, -0.2, -0.8], requires_grad=True)
rewards = torch.tensor([1.0, 0.0, 2.0, 0.5])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', rloo_loss(logps, rewards, group_ids).item())

In [ ]:
from torch_judge import check
check('rloo_loss')